# OMNIVIS — Global Deployment on Google Colab (Free GPU)

This notebook runs the OMNIVIS backend on Colab's free GPU and exposes it globally via ngrok.

### Steps:
1. Set your ngrok auth token
2. Upload your backend folder to Colab
3. Install dependencies
4. Run the server
5. Copy the ngrok URL and configure your frontend


## Step 1: Get your FREE ngrok Auth Token

1. Go to https://dashboard.ngrok.com/signup (FREE)
2. After signing up, go to https://dashboard.ngrok.com/get-started/your-authtoken
3. Copy your auth token
4. Paste it below in the `NGROK_AUTHTOKEN` variable

In [ ]:
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

## Step 2: Upload Backend Files

Upload your backend folder to Colab:
1. Click the **folder icon** on the left sidebar
2. Upload the entire `backend` folder from your OMNIVIS project
3. Or run the cell below to download from GitHub

In [ ]:
# Option A: Download from GitHub (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/OMNIVIS.git /content/OMNIVIS
%cd /content/OMNIVIS/backend

# Option B: If you uploaded manually, just navigate to it
# %cd /content/backend

## Step 3: Install Dependencies

In [ ]:
!pip install fastapi uvicorn[standard] websockets python-multipart
!pip install opencv-python-headless numpy
!pip install scikit-learn psutil Pillow pydantic
!pip install ultralytics
!pip install torch torchvision
!pip install mediapipe
!pip install pyngrok

## Step 4: Start Server & Expose via ngrok

In [ ]:
from pyngrok import ngrok
import threading
import time

# Set ngrok auth token
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🌍 YOUR OMNIVIS BACKEND URL: {public_url}")
print(f"{'='*60}\n")

# Start FastAPI server in background
import uvicorn

def run_server():
    uvicorn.run("main:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server is starting...")
print(f"Keep this notebook running to keep the server alive!")

## Step 5: Copy the ngrok URL above

Use the URL shown above (looks like `https://xxxx.ngrok-free.app`) as your `VITE_BACKEND_URL` when deploying the frontend.

### Deploy Frontend to Vercel (Free):

```bash
# In your frontend directory
npm install -g vercel
vercel --build-env VITE_BACKEND_URL=https://xxxx.ngrok-free.app
```

Or set it in Vercel dashboard:
1. Go to vercel.com → Your Project → Settings → Environment Variables
2. Add `VITE_BACKEND_URL` with your ngrok URL
3. Redeploy

## Important Notes

### ⚠️ Colab Limitations:
- **Sessions expire**: Free tier disconnects after ~12 hours of inactivity
- **Must keep tab open**: Don't close the browser tab running Colab
- **GPU not guaranteed**: Free tier may not always assign a GPU
- **Restart needed**: If disconnected, re-run all cells

### 🔄 Auto-Reconnect Trick:
Paste this in browser console to prevent auto-disconnect:
```javascript
function ClickConnect(){
  document.querySelector('colab-toolbar-button').click();
}
setInterval(ClickConnect, 60000);
```

### 🌐 ngrok Free Tier:
- URL changes every restart (free tier)
- Must update frontend `VITE_BACKEND_URL` each time
- Rate limits apply

### 🚀 For Permanent Hosting (Still Free):
Consider Oracle Cloud Always Free tier with a VM. It runs 24/7 but has no GPU.